<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/APRIORI_EXP6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!pip install -q mlxtend
import pandas as pd, time
from itertools import combinations

# Download grocery dataset (each line = one customer's basket)
!wget -q https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv -O groceries.csv

# Each transaction = set of items
transactions = [set(line.strip().split(",")) for line in open("groceries.csv")]
n = len(transactions)

min_sup, min_conf = 0.02, 0.2   # minimum support 2%, minimum confidence 20%
print(n, "transactions")
print(transactions[:3])

9835 transactions
[{'citrus fruit', 'semi-finished bread', 'ready soups', 'margarine'}, {'coffee', 'yogurt', 'tropical fruit'}, {'whole milk'}]


In [ ]:
# Support = fraction of transactions containing the itemset
def support(itemset):
    return sum(itemset <= t for t in transactions) / n

start = time.time()

# Level 1: frequent single items
items = {i for t in transactions for i in t}
L = {frozenset([i]): support({i}) for i in items}
L = {k: v for k, v in L.items() if v >= min_sup}
all_freq = dict(L)

# Level 2, 3, ...: join itemsets, keep only the frequent ones
k = 2
while L:
    candidates = {a | b for a in L for b in L if len(a | b) == k}
    L = {c: support(c) for c in candidates}
    L = {c: s for c, s in L.items() if s >= min_sup}
    all_freq.update(L)
    k += 1

t_scratch = time.time() - start
print("Frequent itemsets:", len(all_freq), "| Time:", round(t_scratch, 2), "s")

# Top 10 itemsets
for s, v in sorted(all_freq.items(), key=lambda x: -x[1])[:10]:
    print(set(s), round(v, 4))

Frequent itemsets: 122 | Time: 2.99 s
{'whole milk'} 0.2555
{'other vegetables'} 0.1935
{'rolls/buns'} 0.1839
{'soda'} 0.1744
{'yogurt'} 0.1395
{'bottled water'} 0.1105
{'root vegetables'} 0.109
{'tropical fruit'} 0.1049
{'shopping bags'} 0.0985
{'sausage'} 0.094


In [ ]:
rules = []
for itemset, sup in all_freq.items():
    if len(itemset) < 2:
        continue
    for r in range(1, len(itemset)):
        for A in combinations(itemset, r):
            A = frozenset(A)
            B = itemset - A
            conf = sup / all_freq[A]              # confidence of A -> B
            if conf >= min_conf:
                lift = conf / all_freq[B]
                rules.append((set(A), set(B), sup, conf, lift))

rules_scratch = pd.DataFrame(rules, columns=["A", "B", "support", "confidence", "lift"])
print("Strong rules:", len(rules_scratch))
rules_scratch.sort_values("lift", ascending=False).head(10)

Strong rules: 73


,A,B,support,confidence,lift
68,"{other vegetables, whole milk}",{root vegetables},0.023183,0.309783,2.842082
66,{root vegetables},"{other vegetables, whole milk}",0.023183,0.212687,2.842082
23,{pip fruit},{tropical fruit},0.020437,0.270161,2.574648
69,"{root vegetables, whole milk}",{other vegetables},0.023183,0.474012,2.449770
63,{other vegetables},{root vegetables},0.047382,0.244877,2.246605
64,{root vegetables},{other vegetables},0.047382,0.434701,2.246605
71,"{other vegetables, whole milk}",{yogurt},0.022267,0.297554,2.132979
11,{whipped/sour cream},{other vegetables},0.028876,0.402837,2.081924
26,{whipped/sour cream},{yogurt},0.020742,0.289362,2.074251
72,"{yogurt, whole milk}",{other vegetables},0.022267,0.397459,2.054131


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Convert to True/False table
te = TransactionEncoder()
df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

start = time.time()
freq_lib = apriori(df, min_support=min_sup, use_colnames=True)
t_lib = time.time() - start

rules_lib = association_rules(freq_lib, metric="confidence", min_threshold=min_conf)
print("Frequent itemsets:", len(freq_lib), "| Rules:", len(rules_lib))
rules_lib[["antecedents", "consequents", "support", "confidence", "lift"]].sort_values("lift", ascending=False).head(10)

Frequent itemsets: 122 | Rules: 73


,antecedents,consequents,support,confidence,lift
69,(root vegetables),"(other vegetables, whole milk)",0.023183,0.212687,2.842082
67,"(other vegetables, whole milk)",(root vegetables),0.023183,0.309783,2.842082
40,(pip fruit),(tropical fruit),0.020437,0.270161,2.574648
68,"(root vegetables, whole milk)",(other vegetables),0.023183,0.474012,2.449770
27,(other vegetables),(root vegetables),0.047382,0.244877,2.246605
28,(root vegetables),(other vegetables),0.047382,0.434701,2.246605
71,"(other vegetables, whole milk)",(yogurt),0.022267,0.297554,2.132979
32,(whipped/sour cream),(other vegetables),0.028876,0.402837,2.081924
63,(whipped/sour cream),(yogurt),0.020742,0.289362,2.074251
72,"(yogurt, whole milk)",(other vegetables),0.022267,0.397459,2.054131


In [ ]:
pd.DataFrame({
    "Method": ["From scratch", "mlxtend"],
    "Frequent itemsets": [len(all_freq), len(freq_lib)],
    "Rules": [len(rules_scratch), len(rules_lib)],
    "Time (s)": [round(t_scratch, 2), round(t_lib, 2)],
})

,Method,Frequent itemsets,Rules,Time (s)
0,From scratch,122,73,2.99
1,mlxtend,122,73,0.31


In [ ]:
from math import sqrt
from scipy.stats import chi2_contingency

# mc = both items, mnc = only first, nmc = only second, nmnc = neither
def measures(mc, mnc, nmc, nmnc):
    N = mc + mnc + nmc + nmnc
    pA, pB, pAB = (mc + mnc) / N, (mc + nmc) / N, mc / N
    return {
        "chi2":     round(chi2_contingency([[mc, mnc], [nmc, nmnc]], correction=False)[0]),
        "lift":     round(pAB / (pA * pB), 2),
        "all_conf": round(pAB / max(pA, pB), 2),
        "max_conf": round(max(pAB / pA, pAB / pB), 2),
        "Kulc":     round((pAB / pA + pAB / pB) / 2, 2),
        "cosine":   round(pAB / sqrt(pA * pB), 2),
    }

data = {
    "D1": (10000, 1000, 1000, 100000),
    "D2": (10000, 1000, 1000, 100),
    "D3": (100, 1000, 1000, 100000),
    "D4": (1000, 1000, 1000, 100000),
    "D5": (1000, 100, 10000, 100000),
    "D6": (1000, 10, 100000, 100000),
}
pd.DataFrame({k: measures(*v) for k, v in data.items()}).T

,chi2,lift,all_conf,max_conf,Kulc,cosine
D1,90557.0,9.26,0.91,0.91,0.91,0.91
D2,0.0,1.00,0.91,0.91,0.91,0.91
D3,670.0,8.44,0.09,0.09,0.09,0.09
D4,24740.0,25.75,0.50,0.50,0.50,0.50
D5,8173.0,9.18,0.09,0.91,0.50,0.29
D6,966.0,1.97,0.01,0.99,0.50,0.10
